# LAYA hallucination detection on saved TriviaQA outputs (CPU)

This is a separate CPU-only run. It reuses only the completed Qwen TriviaQA generations and BLEURT-20 scores from the Version 6 artifact dataset. LAYA sees the question and Qwen answer; references and BLEURT are joined only for evaluation. Attach `nvthaai/sala-g14-6-source-triviaqa-version-6-artifacts`, enable Internet, and leave the accelerator set to **None**.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 24d39b6
!pip install -q "git+https://github.com/NandhaKishorM/laya.git"

In [ ]:
from pathlib import Path
SOURCE_NAME = "sala_qwen25_g14_6src_triviaqa_v1"
artifact_dirs = [
    path for path in Path("/kaggle/input").rglob(SOURCE_NAME)
    if (path / "generation/triviaqa/rows.jsonl").is_file()
    and (path / "bleurt/triviaqa/scores.jsonl").is_file()
]
if len(artifact_dirs) != 1:
    raise RuntimeError(f"Expected one completed artifact directory, found: {artifact_dirs}")
ARTIFACT_DIR = artifact_dirs[0]
print(f"Using TriviaQA generations and BLEURT scores from {ARTIFACT_DIR}")

In [ ]:
import subprocess
import sys

OUTPUT_DIR = Path("/kaggle/working/laya_triviaqa_cpu")
command = [
    sys.executable, "/kaggle/working/SALA/laya_triviaqa.py",
    "--artifact-dir", str(ARTIFACT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--batch-size", "8",
    "--threshold", "0.5",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd

results = pd.read_csv(OUTPUT_DIR / "results.csv")
display(results.head())
print(json.dumps(json.loads((OUTPUT_DIR / "summary.json").read_text()), indent=2))
print(f"Saved {len(results):,} rows to {OUTPUT_DIR}")